# Assignment: RNN, LSTM, and GRU for Text Generation

## Overview
In this assignment, you will implement three types of recurrent neural networks for character-level text generation using Shakespeare's works. We will:
1. Load and preprocess text data
2. Build Simple RNN, LSTM, and GRU models
3. Train and compare their performance
4. Generate text with each model
5. Analyze which architecture works best

Let's get started!

In [ ]:
# Import Required Libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## Task 1: Load the Dataset

**Objective:** Load the Shakespeare text dataset from TensorFlow/Keras.

**What we're doing:**
- Downloading Shakespeare's complete works text corpus
- This dataset contains thousands of lines from various Shakespeare plays
- Perfect for character-level text generation tasks

**Why:** RNNs excel at learning patterns in sequential data. Shakespeare's text provides rich linguistic patterns for the models to learn from.

In [ ]:
# Task 1: Load the Dataset
# Download Shakespeare dataset from TensorFlow
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

# Read the text file
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

# Display basic information
print(f"Dataset loaded successfully!")
print(f"Total characters in dataset: {len(text)}")
print(f"\nFirst 500 characters of the text:")
print("=" * 80)
print(text[:500])
print("=" * 80)

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Dataset loaded successfully!
Total characters in dataset: 1115394

First 500 characters of the text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


## Task 2: Preprocess the Data

**Objective:** Convert text into numerical sequences that RNNs can process.

**What we're doing:**
- Create a vocabulary: a set of all unique characters in the text
- Map each character to a unique integer (character → index)
- Convert entire text into integer sequences

**Why:** Neural networks process numerical data, not text. We need to encode characters as numbers for the models to learn.

In [ ]:
# Task 2: Preprocess the Data

# Step 1: Create vocabulary (unique characters)
vocab = sorted(set(text))
vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size}")
print(f"Unique characters: {''.join(vocab)}")

# Step 2: Create character to index and index to character mappings
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

# Display sample mappings
print("\nSample character-to-index mappings:")
for i, char in enumerate(list(vocab)[:10]):
    print(f"  '{char}' → {char_to_idx[char]}")

# Step 3: Convert entire text into integer sequence
text_as_int = np.array([char_to_idx[c] for c in text])
print(f"\nText converted to integers. First 100 values:")
print(text_as_int[:100])

# Verify by converting back
print(f"\nVerification - convert back to text:")
print(''.join(idx_to_char[i] for i in text_as_int[:100]))

Vocabulary size: 65
Unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Sample character-to-index mappings:
  '
' → 0
  ' ' → 1
  '!' → 2
  '$' → 3
  '&' → 4
  ''' → 5
  ',' → 6
  '-' → 7
  '.' → 8
  '3' → 9

Text converted to integers. First 100 values:
[18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56 43  1 61 43
  1 54 56 53 41 43 43 42  1 39 52 63  1 44 59 56 58 46 43 56  6  1 46 43
 39 56  1 51 43  1 57 54 43 39 49  8  0  0 13 50 50 10  0 31 54 43 39 49
  6  1 57 54 43 39 49  8  0  0 18 47 56 57 58  1 15 47 58 47 64 43 52 10
  0 37 53 59]

Verification - convert back to text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


## Task 3: Create Input Sequences

**Objective:** Break the text into sequences for supervised learning.

**What we're doing:**
- Choose a sequence length (100 characters in this case)
- Create input-output pairs: input is 100 chars, output is the next character
- Use TensorFlow's Dataset API for efficient batch processing

**Why:** RNNs learn by predicting the next character given a sequence. Each training example has an input sequence and target output.

In [ ]:
# Task 3: Create Input Sequences

# Parameters
seq_length = 100  # Length of input sequence
batch_size = 64   # Number of sequences per batch
buffer_size = 10000  # Shuffle buffer size

# Create sequences: input is seq_length chars, target is the next char
sequences = []
targets = []

for i in range(len(text_as_int) - seq_length):
    sequences.append(text_as_int[i:i + seq_length])
    targets.append(text_as_int[i + seq_length])

sequences = np.array(sequences)
targets = np.array(targets)

print(f"Number of training sequences: {len(sequences)}")
print(f"Sequence shape: {sequences.shape}")
print(f"Target shape: {targets.shape}")

# Example of one sequence
print(f"\nExample sequence (first 20 chars):")
print("Input sequence indices:", sequences[0][:20])
print("Input sequence text:", ''.join(idx_to_char[i] for i in sequences[0]))
print(f"Target index: {targets[0]}")
print(f"Target character: '{idx_to_char[targets[0]]}'")

# Create TensorFlow Dataset
dataset = tf.data.Dataset.from_tensor_slices((sequences, targets))
dataset = dataset.shuffle(buffer_size).batch(batch_size).prefetch(tf.data.AUTOTUNE)

print(f"\nDataset created successfully!")
print(f"Batch size: {batch_size}")
print(f"Total batches: {len(sequences) // batch_size}")

Number of training sequences: 1115294
Sequence shape: (1115294, 100)
Target shape: (1115294,)

Example sequence (first 20 chars):
Input sequence indices: [18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56]
Input sequence text: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
Target index: 1
Target character: ' '

Dataset created successfully!
Batch size: 64
Total batches: 17426


## Task 4: Build the Simple RNN Model

**Objective:** Create a basic RNN (Vanilla RNN) for character prediction.

**Architecture:**
1. **Embedding Layer:** Converts character indices to dense vectors (65 vocab size → 256 dimensions)
2. **SimpleRNN Layer:** Processes sequences and captures patterns (128 units)
3. **Dense Output Layer:** Predicts probability for each character (65 outputs)

**Key Concept:** SimpleRNN maintains a hidden state that's updated at each time step. However, it struggles with long-term dependencies (vanishing gradient problem).

**Why we need the other architectures:** LSTM and GRU were designed to fix the vanishing gradient problem.

In [ ]:
# Task 4: Build the Simple RNN Model

rnn_model = tf.keras.Sequential([
    # Embedding layer: converts character indices to dense vectors
    tf.keras.layers.Embedding(vocab_size, 256, input_length=seq_length),
    
    # SimpleRNN layer: processes sequences
    tf.keras.layers.SimpleRNN(128, return_sequences=False),
    
    # Dense output layer with softmax for probability distribution
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

# Compile the model
rnn_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Display model architecture
print("=" * 80)
print("SIMPLE RNN MODEL")
print("=" * 80)
rnn_model.summary()
print("\nModel built successfully!")

SIMPLE RNN MODEL


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Model built successfully!


## Task 5: Build the LSTM Model

**Objective:** Create an LSTM (Long Short-Term Memory) model for character prediction.

**Architecture:** Same structure as RNN but with LSTM layer instead of SimpleRNN.

**Key Concept:** LSTM uses three gates (input, forget, output) to control information flow:
- **Forget Gate:** Decides what information to discard
- **Input Gate:** Decides what new information to add
- **Output Gate:** Decides what information to output

**Advantage:** LSTMs can learn long-term dependencies better than SimpleRNNs. They solve the vanishing gradient problem with their gate mechanism.

**Trade-off:** LSTMs are more computationally expensive than SimpleRNNs.

In [ ]:
# Task 5: Build the LSTM Model

lstm_model = tf.keras.Sequential([
    # Embedding layer: converts character indices to dense vectors
    tf.keras.layers.Embedding(vocab_size, 256, input_length=seq_length),
    
    # LSTM layer: processes sequences with gated mechanisms
    tf.keras.layers.LSTM(128, return_sequences=False),
    
    # Dense output layer with softmax for probability distribution
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

# Compile the model
lstm_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Display model architecture
print("=" * 80)
print("LSTM MODEL")
print("=" * 80)
lstm_model.summary()
print("\nModel built successfully!")

LSTM MODEL


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Model built successfully!


## Task 6: Build the GRU Model

**Objective:** Create a GRU (Gated Recurrent Unit) model for character prediction.

**Architecture:** Same as RNN and LSTM but with GRU layer.

**Key Concept:** GRU is a simplified version of LSTM with only two gates:
- **Reset Gate:** Controls how much of the previous state to retain
- **Update Gate:** Controls how much of the new state to use

**Advantage over LSTM:**
- Fewer parameters → faster training
- Similar performance on many tasks
- Less prone to overfitting

**When to use:** GRU is great when you have limited computational resources or smaller datasets. LSTM is better for very long sequences or complex patterns.

In [ ]:
# Task 6: Build the GRU Model

gru_model = tf.keras.Sequential([
    # Embedding layer: converts character indices to dense vectors
    tf.keras.layers.Embedding(vocab_size, 256, input_length=seq_length),
    
    # GRU layer: processes sequences with simplified gate mechanism
    tf.keras.layers.GRU(128, return_sequences=False),
    
    # Dense output layer with softmax for probability distribution
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

# Compile the model
gru_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Display model architecture
print("=" * 80)
print("GRU MODEL")
print("=" * 80)
gru_model.summary()
print("\nModel built successfully!")

GRU MODEL


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Model built successfully!


## Task 7: Train All Three Models

**Objective:** Train each model on the Shakespeare dataset.

**Training Details:**
- **Loss Function:** Sparse Categorical Crossentropy (suitable for multi-class classification)
- **Optimizer:** Adam (adaptive learning rate)
- **Epochs:** 5 (for demonstration; production would use more)
- **Validation Split:** 20% (to monitor overfitting)

**What to expect:**
- Training loss should decrease as the model learns
- Validation loss helps detect overfitting
- Accuracy indicates percentage of correct character predictions

In [2]:
# Task 7: Train All Three Models

# Training parameters
epochs = 5
verbose = 1

print("="*80)
print("TRAINING SIMPLE RNN MODEL")
print("="*80)
rnn_history = rnn_model.fit(
    dataset,
    epochs=epochs,
    verbose=verbose
)

print("\n" + "="*80)
print("TRAINING LSTM MODEL")
print("="*80)
lstm_history = lstm_model.fit(
    dataset,
    epochs=epochs,
    verbose=verbose
)

print("\n" + "="*80)
print("TRAINING GRU MODEL")
print("="*80)
gru_history = gru_model.fit(
    dataset,
    epochs=epochs,
    verbose=verbose
)

print("\n" + "="*80)
print("ALL MODELS TRAINED SUCCESSFULLY!")
print("="*80)

LoadError: MethodError: no method matching *(::String, ::Int64)
The function `*` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  *(::Any, ::Any, [91m::Any[39m, [91m::Any...[39m)
[0m[90m   @[39m [90mBase[39m [90m[4moperators.jl:642[24m[39m
[0m  *([91m::BigFloat[39m, ::Union{Int16, Int32, Int64, Int8})
[0m[90m   @[39m [90mBase[39m [90m[4mmpfr.jl:592[24m[39m
[0m  *([91m::BigInt[39m, ::Union{Int16, Int32, Int64, Int8})
[0m[90m   @[39m [90mBase[39m [90m[4mgmp.jl:563[24m[39m
[0m  ...


## Task 8: Compare Model Performance

**Objective:** Analyze and visualize the performance metrics of all three models.

**What we'll examine:**
1. **Training Loss:** How well each model learned the training data
2. **Accuracy:** Percentage of correct predictions
3. **Convergence Speed:** Which model learned fastest
4. **Overfitting:** Difference between training and validation metrics

**Key Metrics to Compare:**
- Convergence speed: Faster is better
- Final training loss: Lower is better
- Stability: Less fluctuation is better

In [ ]:
# Task 8: Compare Model Performance

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Loss Comparison
axes[0].plot(rnn_history.history['loss'], label='RNN - Training Loss', marker='o', linewidth=2)
axes[0].plot(lstm_history.history['loss'], label='LSTM - Training Loss', marker='s', linewidth=2)
axes[0].plot(gru_history.history['loss'], label='GRU - Training Loss', marker='^', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Accuracy Comparison
axes[1].plot(rnn_history.history['accuracy'], label='RNN - Training Accuracy', marker='o', linewidth=2)
axes[1].plot(lstm_history.history['accuracy'], label='LSTM - Training Accuracy', marker='s', linewidth=2)
axes[1].plot(gru_history.history['accuracy'], label='GRU - Training Accuracy', marker='^', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training Accuracy Comparison', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print Detailed Comparison
print("\n" + "="*80)
print("PERFORMANCE COMPARISON SUMMARY")
print("="*80)

print("\nRNN Model:")
print(f"  Initial Loss: {rnn_history.history['loss'][0]:.4f}")
print(f"  Final Loss:   {rnn_history.history['loss'][-1]:.4f}")
print(f"  Initial Accuracy: {rnn_history.history['accuracy'][0]:.4f}")
print(f"  Final Accuracy:   {rnn_history.history['accuracy'][-1]:.4f}")

print("\nLSTM Model:")
print(f"  Initial Loss: {lstm_history.history['loss'][0]:.4f}")
print(f"  Final Loss:   {lstm_history.history['loss'][-1]:.4f}")
print(f"  Initial Accuracy: {lstm_history.history['accuracy'][0]:.4f}")
print(f"  Final Accuracy:   {lstm_history.history['accuracy'][-1]:.4f}")

print("\nGRU Model:")
print(f"  Initial Loss: {gru_history.history['loss'][0]:.4f}")
print(f"  Final Loss:   {gru_history.history['loss'][-1]:.4f}")
print(f"  Initial Accuracy: {gru_history.history['accuracy'][0]:.4f}")
print(f"  Final Accuracy:   {gru_history.history['accuracy'][-1]:.4f}")

# Analysis
print("\n" + "="*80)
print("ANALYSIS AND OBSERVATIONS")
print("="*80)

# Which model converges fastest
losses = {
    'RNN': rnn_history.history['loss'][-1],
    'LSTM': lstm_history.history['loss'][-1],
    'GRU': gru_history.history['loss'][-1]
}
best_loss_model = min(losses, key=losses.get)
print(f"\n1. Lowest Final Loss: {best_loss_model} ({losses[best_loss_model]:.4f})")

# Which has highest accuracy
accuracies = {
    'RNN': rnn_history.history['accuracy'][-1],
    'LSTM': lstm_history.history['accuracy'][-1],
    'GRU': gru_history.history['accuracy'][-1]
}
best_acc_model = max(accuracies, key=accuracies.get)
print(f"2. Highest Final Accuracy: {best_acc_model} ({accuracies[best_acc_model]:.4f})")

# Convergence stability
print(f"\n3. Training Stability (Loss variance):")
print(f"   RNN Loss variance: {np.var(rnn_history.history['loss']):.6f}")
print(f"   LSTM Loss variance: {np.var(lstm_history.history['loss']):.6f}")
print(f"   GRU Loss variance: {np.var(gru_history.history['loss']):.6f}")

## Task 9: Generate Text with Each Model

**Objective:** Use each trained model to generate Shakespeare-like text.

**How Text Generation Works:**
1. Start with an initial sequence (e.g., "ROMEO:")
2. Pass it through the model to predict the next character
3. Append the predicted character to the sequence
4. Repeat steps 2-3 for N iterations

**Key Concepts:**
- **Sampling:** Use `tf.random.categorical` to sample from probability distribution (adds diversity)
- **Temperature:** Controls randomness (low = deterministic, high = more random)
- **Without sampling:** Always pick highest probability (deterministic, less creative)

**Quality Indicators:**
- Realistic character sequences
- Proper grammar patterns
- Coherent word formations

In [ ]:
# Task 9: Generate Text with Each Model

def generate_text(model, start_string, num_generate=200, temperature=1.0):
    """
    Generate text using a trained model.
    
    Parameters:
    - model: Trained RNN/LSTM/GRU model
    - start_string: Initial text to start generation
    - num_generate: Number of characters to generate
    - temperature: Controls randomness (1.0 = normal, higher = more random)
    
    Returns:
    - Generated text string
    """
    # Convert start_string to integers
    input_seq = [char_to_idx[char] for char in start_string]
    
    # Generate num_generate characters
    generated_text = start_string
    
    for _ in range(num_generate):
        # Ensure we have exactly seq_length input
        if len(input_seq) < seq_length:
            # Pad with zeros at the beginning if needed
            current_input = [0] * (seq_length - len(input_seq)) + input_seq
        else:
            # Take the last seq_length characters
            current_input = input_seq[-seq_length:]
        
        # Reshape for model input
        current_input = np.array([current_input])
        
        # Predict next character probabilities
        predictions = model.predict(current_input, verbose=0)
        
        # Apply temperature scaling
        predictions = np.log(predictions + 1e-10) / temperature
        predictions = np.exp(predictions) / np.sum(np.exp(predictions))
        
        # Sample from the probability distribution
        predicted_id = np.random.choice(vocab_size, p=predictions[0])
        
        # Convert to character
        predicted_char = idx_to_char[predicted_id]
        
        # Append to generated text
        generated_text += predicted_char
        input_seq.append(predicted_id)
    
    return generated_text

# Generate text from each model
start_text = "ROMEO:"
num_chars = 200

print("="*80)
print("TEXT GENERATION RESULTS")
print("="*80)

print(f"\nStarting string: '{start_text}'")
print(f"Characters to generate: {num_chars}")

print("\n" + "-"*80)
print("RNN GENERATED TEXT (Temperature=1.0)")
print("-"*80)
rnn_generated = generate_text(rnn_model, start_text, num_chars, temperature=1.0)
print(rnn_generated)

print("\n" + "-"*80)
print("LSTM GENERATED TEXT (Temperature=1.0)")
print("-"*80)
lstm_generated = generate_text(lstm_model, start_text, num_chars, temperature=1.0)
print(lstm_generated)

print("\n" + "-"*80)
print("GRU GENERATED TEXT (Temperature=1.0)")
print("-"*80)
gru_generated = generate_text(gru_model, start_text, num_chars, temperature=1.0)
print(gru_generated)

print("\n" + "="*80)
print("TEXT GENERATION COMPARISON")
print("="*80)

# Analyze generated text quality
def analyze_text_quality(text):
    """Simple quality metrics for generated text"""
    # Count punctuation
    punctuation = sum(1 for c in text if c in '.,!?;:')
    # Count spaces
    spaces = text.count(' ')
    # Count common patterns
    
    return {
        'length': len(text),
        'punctuation': punctuation,
        'spaces': spaces,
        'avg_word_length': len(text) / max(1, spaces)
    }

rnn_quality = analyze_text_quality(rnn_generated)
lstm_quality = analyze_text_quality(lstm_generated)
gru_quality = analyze_text_quality(gru_generated)

print(f"\nRNN Text Quality: {rnn_quality}")
print(f"LSTM Text Quality: {lstm_quality}")
print(f"GRU Text Quality: {gru_quality}")

## Task 10: Final Reflection

**Your Task:** Answer the following questions based on your observations:

1. **Text Generation Quality:**
   - How does the quality of generated text differ between the three models?
   - Which model produces more coherent and realistic Shakespeare-like text?
   - Which model produces more random/nonsensical output?

2. **Architecture Performance:**
   - Based on loss/accuracy plots and generated text, which architecture performs best?
   - Why do you think that architecture performed better?
   - What are the strengths of each model?

3. **Trade-offs:**
   - What are the computational trade-offs between RNN, LSTM, and GRU?
   - How do overfitting tendencies differ between the models?
   - When would you choose GRU over LSTM and vice versa?

**Write your reflection below (3-4 paragraphs):**

In [ ]:
# Task 10: Final Reflection - Your Analysis

reflection = """
REFLECTION: RNN vs LSTM vs GRU for Text Generation

TEXT GENERATION QUALITY ANALYSIS:
[Write your observations here about how the text quality differs between models]
- Which model produced the most coherent text?
- Which model seemed to understand grammar and punctuation better?
- Did you notice any patterns in the generated text?

ARCHITECTURE PERFORMANCE:
[Write your analysis based on the metrics and generated text]
- What does the loss/accuracy comparison tell us?
- Did the model with the lowest loss produce the best text?
- Why might LSTM or GRU outperform SimpleRNN?

COMPUTATIONAL TRADE-OFFS:
[Compare the trade-offs between the three architectures]
- SimpleRNN is faster but may struggle with long-term dependencies
- LSTM has more parameters but handles long sequences better
- GRU is between RNN and LSTM in terms of complexity and performance
- For character-level text generation, which trade-off is worth it?

CONCLUSION:
[Summarize which architecture is best for this task and why]
- Based on all evidence (loss, accuracy, generated text quality), recommend one model
- Explain when you'd choose each model in real-world scenarios
"""

print("="*80)
print("REFLECTION TEMPLATE")
print("="*80)
print(reflection)

# Summary statistics
print("\n" + "="*80)
print("QUICK SUMMARY STATISTICS")
print("="*80)

models_info = {
    'SimpleRNN': {
        'final_loss': rnn_history.history['loss'][-1],
        'final_acc': rnn_history.history['accuracy'][-1],
        'params': rnn_model.count_params()
    },
    'LSTM': {
        'final_loss': lstm_history.history['loss'][-1],
        'final_acc': lstm_history.history['accuracy'][-1],
        'params': lstm_model.count_params()
    },
    'GRU': {
        'final_loss': gru_history.history['loss'][-1],
        'final_acc': gru_history.history['accuracy'][-1],
        'params': gru_model.count_params()
    }
}

print(f"\n{'Model':<15} {'Final Loss':<15} {'Final Accuracy':<15} {'Parameters':<15}")
print("-" * 60)
for model_name, metrics in models_info.items():
    print(f"{model_name:<15} {metrics['final_loss']:<15.4f} {metrics['final_acc']:<15.4f} {metrics['params']:<15,}")

print("\n" + "="*80)
print("ASSIGNMENT COMPLETE!")
print("="*80)
print("\nYou have successfully:")
print("✓ Loaded and preprocessed Shakespeare dataset")
print("✓ Built three RNN architectures (SimpleRNN, LSTM, GRU)")
print("✓ Trained and compared all models")
print("✓ Generated text with each model")
print("✓ Analyzed and compared performance metrics")
print("\nNow write your reflection in the cell above based on your observations!")

## Key Takeaways

### Architecture Comparison Table

| Feature | SimpleRNN | LSTM | GRU |
|---------|-----------|------|-----|
| **Complexity** | Low | High | Medium |
| **Parameters** | Fewer | Most | Medium |
| **Long-term Dependencies** | Poor | Excellent | Good |
| **Training Speed** | Fast | Slow | Medium |
| **Overfitting Risk** | Lower | Higher | Medium |
| **Best For** | Simple patterns | Complex sequences | Resource-limited |

### Important Concepts

1. **Vanishing Gradient Problem:** SimpleRNN struggles to learn long-term dependencies because gradients become too small during backpropagation.

2. **LSTM Solution:** Uses memory cells and three gates (input, forget, output) to maintain gradient flow through long sequences.

3. **GRU Advantage:** Simplifies LSTM with only two gates (reset, update) while maintaining similar performance with fewer parameters.

4. **Character-Level Generation:** Predicting the next character is a character-level language modeling task—each model learns character patterns and transitions.

5. **Text Quality vs Loss:** Lower loss doesn't always mean better generated text. A model can memorize training data (high loss) but still generate incoherent text.

### Real-World Applications

- **RNN:** Time-series forecasting, simple sequential data
- **LSTM:** Machine translation, speech recognition, long document analysis
- **GRU:** When computational resources are limited, mobile applications, real-time processing

---

**Now run all cells to execute the full assignment and see the results!**